# BA-Net single-fold renal-vessel inference

## Goal

Run the authors' published BA-Net fold-0 checkpoint on one 3D CT using a free Colab GPU, then download a small, identity-checked package for the Renal CT Research Explorer. This notebook does **not** download or redistribute the KiPA22 training dataset.

The checkpoint was trained on cropped KiPA22 CTA. Results from KiTS23, venous-phase, non-contrast, or differently reconstructed CT are out of domain, research-only, and not suitable for diagnosis or surgical decisions.

## Setup

In Colab choose **Runtime → Change runtime type → T4 GPU**. Set `CASE_ID` to exactly the case shown in Streamlit. Fold 0 is the lightweight default; enabling test-time augmentation is slower and does not remove domain shift.

In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

CASE_ID = "case_00015"  # Must exactly match the selected Streamlit case
FOLD = 0                   # Published folds: 0, 1, 2, 3
DISABLE_TTA = True         # Faster; set False for mirrored test-time augmentation

IN_COLAB = "google.colab" in sys.modules
RUN_INFERENCE = IN_COLAB
assert 0 <= FOLD <= 3
assert CASE_ID and all(ch.isalnum() or ch in "_.-" for ch in CASE_ID)
print("Mode:", "Colab inference" if RUN_INFERENCE else "local validation (heavy cells skipped)")

In [ ]:
APP_REPOSITORY = "https://github.com/ahmederaky2-cmyk/renal-ct-explorer.git"
if IN_COLAB:
    PROJECT_ROOT = Path("/content/renal-ct-explorer")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", APP_REPOSITORY, str(PROJECT_ROOT)],
            check=True,
        )
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    PROJECT_ROOT = next(
        path for path in candidates if (path / "notebooks/banet_colab.py").is_file()
    )

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
workflow = importlib.import_module("banet_colab")

print("Workflow source:", PROJECT_ROOT)
print("Published weights DOI:", workflow.WEIGHTS_DOI)
print(
    "Full archive:",
    f"{workflow.EXPECTED_ARCHIVE_BYTES / 1024**3:.2f} GiB; only one fold will be fetched",
)

## Steps

### 1. Prepare the official inference code

The notebook clones the authors' `kipa22` branch instead of copying their source into this project. Narrow compatibility adjustments are applied only inside the temporary Colab checkout for current NumPy/PyTorch behavior.

In [ ]:
WORK_ROOT = (
    Path("/content/renal_banet_work")
    if IN_COLAB
    else PROJECT_ROOT / "work/notebook_validation"
)
BANET_ROOT = WORK_ROOT / "BA-Net"
if RUN_INFERENCE:
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    if not BANET_ROOT.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", workflow.BANET_BRANCH,
            workflow.BANET_REPOSITORY, str(BANET_ROOT)
        ], check=True)
    workflow.patch_banet_for_current_colab(BANET_ROOT)
    workflow.install_banet_dependencies(BANET_ROOT)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Select a T4 GPU runtime and reconnect.")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Skipped code installation outside Colab.")

### 2. Upload the selected CT

Upload the exact `imaging.nii.gz` used by the selected Streamlit case. The original compressed bytes are hashed before inference; the app will reject a result made from any other CT. Do not upload identifiable clinical data unless you are authorized to process it in Colab.

In [ ]:
SOURCE_ORIGINAL = None
INPUT_FOLDER = WORK_ROOT / "input"
if RUN_INFERENCE:
    from google.colab import files
    uploaded = files.upload()
    candidates = [(name, data) for name, data in uploaded.items() if name.endswith(".nii.gz")]
    if len(candidates) != 1:
        raise ValueError("Upload exactly one .nii.gz CT image.")
    original_name, original_bytes = candidates[0]
    INPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    SOURCE_ORIGINAL = WORK_ROOT / "source" / original_name
    SOURCE_ORIGINAL.parent.mkdir(parents=True, exist_ok=True)
    SOURCE_ORIGINAL.write_bytes(original_bytes)
    nnunet_input = INPUT_FOLDER / f"{CASE_ID}_0000.nii.gz"
    nnunet_input.write_bytes(original_bytes)
    import nibabel as nib
    image = nib.load(str(SOURCE_ORIGINAL))
    if len(image.shape) != 3:
        raise ValueError(f"Expected one 3D CT, found shape {image.shape}.")
    print("CT:", image.shape, "SHA-256:", workflow.sha256_file(SOURCE_ORIGINAL))
else:
    print("Upload is available when this notebook runs in Colab.")

### 3. Fetch only one published checkpoint fold

The Zenodo release is a 3.7 GB BA-Net + ResUNet ensemble. This step reads its ZIP index remotely and downloads only `plans.pkl` plus BA-Net fold 0—roughly 355 MB compressed. Every extracted member is checked against the archive's CRC and uncompressed size. If the host stops supporting byte ranges or the pinned archive size changes, the notebook stops instead of silently downloading or trusting a different artifact.

In [ ]:
RESULTS_ROOT = WORK_ROOT / "results"
MEMBER_INTEGRITY = {}
if RUN_INFERENCE:
    model_root, MEMBER_INTEGRITY = workflow.install_single_banet_fold(RESULTS_ROOT, FOLD)
    print("Installed model folder:", model_root)
    print("Verified members:", MEMBER_INTEGRITY)
else:
    print("Checkpoint download skipped outside Colab.")

### 4. Run BA-Net inference

The output retains the authors' KiPA label convention: `1=renal vein`, `2=kidney`, `3=renal artery`, `4=tumor`. The Streamlit importer discards non-vessel predictions and maps artery/vein into its common label vocabulary.

In [ ]:
OUTPUT_FOLDER = WORK_ROOT / "prediction"
PREDICTION_PATH = OUTPUT_FOLDER / f"{CASE_ID}.nii.gz"
if RUN_INFERENCE:
    raw_root = WORK_ROOT / "nnUNet_raw"
    preprocessed_root = WORK_ROOT / "nnUNet_preprocessed"
    raw_root.mkdir(parents=True, exist_ok=True)
    preprocessed_root.mkdir(parents=True, exist_ok=True)
    OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
    inference_environment = {
        **os.environ,
        "nnUNet_raw_data_base": str(raw_root),
        "nnUNet_preprocessed": str(preprocessed_root),
        "RESULTS_FOLDER": str(RESULTS_ROOT),
    }
    command = [
        "nnUNet_predict", "-i", str(INPUT_FOLDER), "-o", str(OUTPUT_FOLDER),
        "-t", "1001", "-m", "3d_fullres", "-tr", workflow.TRAINER,
        "-p", workflow.PLANS_IDENTIFIER, "-f", str(FOLD),
        "-chk", "model_final_checkpoint", "--num_threads_preprocessing", "2",
        "--num_threads_nifti_save", "1", "--overwrite_existing"
    ]
    if DISABLE_TTA:
        command.append("--disable_tta")
    subprocess.run(command, check=True, env=inference_environment)
    if not PREDICTION_PATH.is_file():
        raise RuntimeError(f"Expected prediction was not created: {PREDICTION_PATH}")
    print("Prediction:", PREDICTION_PATH)
else:
    print("Inference skipped outside Colab.")

### 5. Build and download the Streamlit package

The package contains only `vessels.nii.gz` and `provenance.json`. It records the exact input hash, spatial geometry, checkpoint source, fold, compatibility checkout, and research-only status.

In [ ]:
PACKAGE_PATH = WORK_ROOT / f"{CASE_ID}_banet_fold{FOLD}.zip"
if RUN_INFERENCE:
    banet_commit = subprocess.check_output(
        ["git", "-C", str(BANET_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
    provenance = workflow.build_prediction_package(
        SOURCE_ORIGINAL, PREDICTION_PATH, PACKAGE_PATH, CASE_ID, FOLD,
        banet_commit, MEMBER_INTEGRITY, DISABLE_TTA
    )
    print("Package:", PACKAGE_PATH, f"({PACKAGE_PATH.stat().st_size / 1024**2:.1f} MiB)")
    print("Labels found:", provenance["source_labels"])
    from google.colab import files
    files.download(str(PACKAGE_PATH))
else:
    print("Package creation skipped outside Colab.")

## Checks

A successful Colab run must show: a CUDA GPU, three CRC-verified checkpoint members, a generated NIfTI prediction, and a downloaded ZIP. The Streamlit app performs a second independent check of case ID, original-file SHA-256, canonical dimensions, affine geometry, allowed labels, and research-only provenance.

In [ ]:
assert workflow.WEIGHTS_DOI == "10.5281/zenodo.7030423"
assert workflow.EXPECTED_ARCHIVE_BYTES == 3_708_026_487
assert workflow.EXPECTED_ARCHIVE_MD5 == "fb22a9730070fdf45ed3923b1d82b4d2"
if RUN_INFERENCE:
    assert PREDICTION_PATH.is_file() and PACKAGE_PATH.is_file()
    assert set(MEMBER_INTEGRITY) == {
        "BA-Net/plans.pkl",
        f"BA-Net/fold_{FOLD}/model_final_checkpoint.model",
        f"BA-Net/fold_{FOLD}/model_final_checkpoint.model.pkl",
    }
    print("Inference workflow checks passed.")
else:
    print("Notebook structure/helper checks passed; GPU inference was intentionally skipped.")

## Next Steps

1. Return to the Streamlit app and expand **Vessel ML prediction**.
2. Upload the downloaded `*_banet_fold0.zip` file.
3. Select **Renal artery** and **Renal vein** in **Structures**.
4. Open **Interactive 3D** and choose **Build interactive 3D**.

Treat the result as an experimental visualization. Before any scientific comparison, validate artery and vein Dice plus centerline/topology metrics on an authorized external CTA cohort.